# TuringBench Ensemble Training
Train a logistic regression ensemble over multiple fine-tuned TuringBench detectors.

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch==2.6.1 torchvision==0.21.1 torchaudio==2.6.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets accelerate scikit-learn pandas joblib

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import os
repo_url = 'https://github.com/vedangvatsa/ai-detection-at-scale.git'
repo_dir = '/kaggle/working/ai-detection-at-scale'
if not os.path.exists(repo_dir):
    !git clone $repo_url $repo_dir
else:
    !git -C $repo_dir pull origin main
print('Repo ready at', repo_dir)

In [ ]:
import os
import glob

# Add model directories here after each fine-tuning run completes.
# Example: ['/kaggle/working/models/turingbench_roberta_large', '/kaggle/working/models/turingbench_deberta_v3_large']

model_dirs = [
    d for d in glob.glob('/kaggle/working/models/*')
    if os.path.isdir(d) and 'turingbench' in d.lower()
]
print('Models found:', model_dirs)

if len(model_dirs) < 2:
    raise ValueError('Need at least 2 fine-tuned models in /kaggle/working/models/')

script = os.path.join(repo_dir, 'scripts', 'ensemble_turingbench.py')
out_dir = '/kaggle/working/models/turingbench_ensemble'
cmd = (
    f"python {script} "
    f"--model_dirs {' '.join(model_dirs)} "
    f"--output_dir {out_dir} "
    f"--max_length 512 "
    f"--batch_size 32"
)
print('Running:', cmd)
!{cmd}

In [ ]:
import shutil
results_src = os.path.join(repo_dir, 'results', 'turingbench_ensemble.csv')
results_dst = '/kaggle/working/results/turingbench_ensemble.csv'
os.makedirs('/kaggle/working/results', exist_ok=True)
if os.path.exists(results_src):
    shutil.copy(results_src, results_dst)
    print('Results copied to', results_dst)
print('Outputs in', out_dir)